https://youtu.be/TcRgkpF4K2M

In [0]:
salary_data = [
    (1, 'Rohan', 5000),
    (2, 'Alex', 6000),
    (3, 'Maryam', 7000)
]

salary_schema = "emp_id int, emp_name string, base_salary int"

salary_df = spark.createDataFrame(data = salary_data, schema = salary_schema)
salary_df.display()

income_data = [
    (1,'Basic', 100),
    (2,'Allowance', 4),
    (3,'Others', 6)
]

income_schema = "id int, income string, percentage int"

income_df = spark.createDataFrame(data = income_data, schema = income_schema)
income_df.display()

deduction_data = [
    (1,'Insurance', 5),
    (2,'Health', 6),
    (3,'House', 4)
]

deduction_schema = "id int, deduction string, percentage int"

deduction_df = spark.createDataFrame(data = deduction_data, schema = deduction_schema)
deduction_df.display()

In [0]:
from pyspark.sql import functions as F

In [0]:
df = salary_df.crossJoin(income_df)
df2= salary_df.crossJoin(deduction_df)

df_final = (
    df.union(df2)
        .withColumn("Amount", F.col("base_salary")*F.col("percentage")/100)
        .select("emp_id","emp_name", "income","Amount")
        .orderBy("emp_id","income")
)

df_final.display()

#df.display()
#df2.display()

In [0]:
df_new = (
    df_final
        .groupBy("emp_name")
        .pivot("income")
        .sum("Amount")
        .withColumn("Gross", F.col("Allowance") + F.col("Basic") + F.col("Others"))
        .withColumn("Total_deduction", F.col("insurance") + F.col("Health") + F.col("House"))
        .withColumn("NET_PAY", F.col("Gross") - F.col("Total_deduction"))
        .select("emp_name","Basic","Allowance","others", "Gross", "insurance","Health","House","Total_deduction", "NET_PAY")
        .orderBy("emp_name")
)

df_new.display()